# Corrección: festivos nacionales frente a autonómicos

La evaluación dirigida a festivos reveló un patrón claro. Los modelos de
aprendizaje automático superan ampliamente a las líneas base en los festivos
nacionales, pero no en los autonómicos:

| Ámbito | Ingenuo semanal | LightGBM | Casos en que gana el ML |
|---|---|---|---|
| Nacional (6 festivos) | 20,20 % | 3,73 % | 6 de 6 |
| Solo Madrid (2 festivos) | 5,94 % | 6,05 % | 1 de 2 |

## La causa

La serie de demanda es **peninsular**, mientras que el calendario empleado
corresponde a la **Comunidad de Madrid**. El 2 de mayo el modelo recibe
`festivo = 1` y anticipa una caída pronunciada, pero el resto del país trabaja
con normalidad y la demanda apenas desciende. El modelo sobrestima el desplome.

## La solución

Una variable binaria no distingue entre un festivo que afecta a todo el país y
otro que afecta al 14 % de la población. Se sustituye por una **intensidad
ponderada por población**: la fracción de habitantes para los que ese día es
festivo.

| Fecha | Festividad | Comunidades | Intensidad |
|---|---|---|---|
| 25-Dic | Natividad del Señor | 19 | 1,000 |
| 28-Mar | Jueves Santo | 17 | 0,725 |
| 25-Jul | Santiago Apóstol | 5 | 0,271 |
| 02-May | Fiesta de la Comunidad de Madrid | 1 | 0,143 |

Obsérvese que el Jueves Santo, clasificado como autonómico por la librería,
afecta en realidad al 72,5 % de la población. Una división binaria
nacional/autonómico también fallaría en ese caso; la intensidad continua lo
resuelve.

En los ocho festivos evaluados, la correlación entre intensidad y error del
modelo es de **−0,46**: cuanto menor es la intensidad, mayor es el error. El
diagnóstico queda confirmado.

---
## Ejecución

Este cuaderno **modifica `datos/dataset_modelado.parquet`** añadiendo las
variables nuevas. No es necesario volver a ejecutar el notebook 01.

In [1]:
import numpy as np
import pandas as pd
import holidays
from pathlib import Path

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

RUTA = RAIZ / "datos" / "dataset_modelado.parquet"
df = pd.read_parquet(RUTA)

# Se guarda el índice original para restaurarlo al final. Convertir a UTC y
# volver desplazaría las fechas una o dos horas y descuadraría los festivos.
INDICE_ORIGINAL = df.index.copy()
if df.index.tz is not None:
    df.index = df.index.tz_localize(None)

print(f"Dataset cargado: {df.shape}")
print(f"Periodo: {df.index.min():%Y-%m-%d} a {df.index.max():%Y-%m-%d}")

Dataset cargado: (26136, 46)
Periodo: 2022-01-08 a 2024-12-31


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# 1 — Intensidad de festivo ponderada por población
# ═══════════════════════════════════════════════════════════════════════════
# Población por comunidad autónoma (INE, en millones). Se emplea para ponderar
# el alcance real de cada festividad sobre la demanda peninsular.
POBLACION = {
    "AN": 8.63, "AR": 1.35, "AS": 1.01, "CB": 0.59, "CE": 0.08,
    "CL": 2.38, "CM": 2.10, "CN": 2.24, "CT": 8.01, "EX": 1.06,
    "GA": 2.70, "IB": 1.23, "MC": 1.57, "MD": 6.95, "ML": 0.09,
    "NC": 0.67, "PV": 2.22, "RI": 0.32, "VC": 5.32,
}
TOTAL_POB = sum(POBLACION.values())

ANIOS = range(df.index.year.min(), df.index.year.max() + 1)

# Calendario de cada comunidad
calendarios = {s: holidays.Spain(years=ANIOS, subdiv=s) for s in POBLACION}

# Intensidad de cada fecha festiva
fechas_festivas = sorted({d for c in calendarios.values() for d in c})
intensidad = {
    d: sum(p for s, p in POBLACION.items() if d in calendarios[s]) / TOTAL_POB
    for d in fechas_festivas
}

print(f"Fechas festivas en alguna comunidad: {len(intensidad)}")

# Asignar a cada hora del dataset
fechas_idx = pd.Series(df.index.date, index=df.index)
df["intensidad_festivo"] = fechas_idx.map(intensidad).fillna(0.0).astype(float)

# Variantes binarias derivadas
df["festivo_nacional"]   = (df["intensidad_festivo"] >= 0.90).astype("int8")
df["festivo_parcial"]    = ((df["intensidad_festivo"] > 0.05) &
                            (df["intensidad_festivo"] < 0.90)).astype("int8")

print(f"\nHoras con festivo nacional  : {df['festivo_nacional'].sum():,}")
print(f"Horas con festivo parcial   : {df['festivo_parcial'].sum():,}")
print(f"Intensidad media (solo >0)  : "
      f"{df.loc[df['intensidad_festivo']>0,'intensidad_festivo'].mean():.3f}")

Fechas festivas en alguna comunidad: 109

Horas con festivo nacional  : 576
Horas con festivo parcial   : 888
Intensidad media (solo >0)  : 0.321


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# 2 — Víspera y día posterior ponderados
# ═══════════════════════════════════════════════════════════════════════════
# El efecto de arrastre de un festivo también depende de su alcance: la víspera
# de Navidad no es comparable a la del 2 de mayo.

serie_int = pd.Series(intensidad)
serie_int.index = pd.to_datetime(serie_int.index)

fechas_norm = pd.Series(df.index.normalize(), index=df.index)

df["intensidad_vispera"] = (fechas_norm + pd.Timedelta(days=1)).map(serie_int).fillna(0.0)
df["intensidad_post"]    = (fechas_norm - pd.Timedelta(days=1)).map(serie_int).fillna(0.0)

# Laborable efectivo: descuenta la fracción de población en festivo
df["laborable_efectivo"] = (
    (df.index.dayofweek < 5).astype(float) * (1 - df["intensidad_festivo"])
)

print("Variables añadidas:\n")
NUEVAS = ["intensidad_festivo", "festivo_nacional", "festivo_parcial",
          "intensidad_vispera", "intensidad_post", "laborable_efectivo"]
print(df[NUEVAS].describe().round(3).T.to_string())

Variables añadidas:

                      count   mean    std  min  25%  50%  75%  max
intensidad_festivo  26136.0  0.032  0.157  0.0  0.0  0.0  0.0  1.0
festivo_nacional    26136.0  0.022  0.147  0.0  0.0  0.0  0.0  1.0
festivo_parcial     26136.0  0.034  0.181  0.0  0.0  0.0  0.0  1.0
intensidad_vispera  26136.0  0.032  0.157  0.0  0.0  0.0  0.0  1.0
intensidad_post     26136.0  0.032  0.157  0.0  0.0  0.0  0.0  1.0
laborable_efectivo  26136.0  0.684  0.459  0.0  0.0  1.0  1.0  1.0


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# 3 — Comprobación sobre los festivos evaluados
# ═══════════════════════════════════════════════════════════════════════════
comprobar = ["2024-05-01", "2024-05-02", "2024-07-25", "2024-08-15",
             "2024-10-12", "2024-11-01", "2024-12-06", "2024-12-25"]

filas = []
for f in comprobar:
    ts = pd.Timestamp(f)
    if ts not in df.index.normalize():
        continue
    fila = df[df.index.normalize() == ts].iloc[0]
    nombre = next((calendarios[s][ts.date()] for s in POBLACION
                   if ts.date() in calendarios[s]), "-")
    filas.append({
        "fecha": f,
        "festividad": nombre[:34],
        "festivo (antes)": int(fila.get("festivo", 0)),
        "intensidad": round(fila["intensidad_festivo"], 3),
        "nacional": int(fila["festivo_nacional"]),
    })

print("Valor de las variables en los festivos evaluados\n")
display(pd.DataFrame(filas))

print("\nLa variable antigua asignaba 1 a todos por igual. La intensidad")
print("distingue el 2 de mayo (0,143) de Navidad (1,000), que es exactamente")
print("la información que faltaba al modelo.")

Valor de las variables en los festivos evaluados



,fecha,festividad,festivo (antes),intensidad,nacional
0,2024-05-01,Fiesta del Trabajo,1,1.000,1
1,2024-05-02,Fiesta de la Comunidad de Madrid,1,0.143,0
2,2024-07-25,Santiago Apóstol,1,0.271,0
3,2024-08-15,Asunción de la Virgen,1,1.000,1
4,2024-10-12,Fiesta Nacional de España,1,1.000,1
5,2024-11-01,Todos los Santos,1,1.000,1
6,2024-12-06,Día de la Constitución Española,1,1.000,1
7,2024-12-25,Natividad del Señor,1,1.000,1



La variable antigua asignaba 1 a todos por igual. La intensidad
distingue el 2 de mayo (0,143) de Navidad (1,000), que es exactamente
la información que faltaba al modelo.


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# 4 — Guardar el dataset actualizado
# ═══════════════════════════════════════════════════════════════════════════
df.index = INDICE_ORIGINAL      # restaura la zona horaria exacta
df.to_parquet(RUTA)

print(f"Guardado en {RUTA}")
print(f"Dimensiones: {df.shape}")
print(f"\nVariables nuevas disponibles para el notebook 03:")
for c in NUEVAS:
    print(f"   {c}")

Guardado en c:\Users\rober\OneDrive - unizar.es\Escritorio\Master\BigData y Ciencia de datos\TFM\GITHUB\repo_github_tfm\datos\dataset_modelado.parquet
Dimensiones: (26136, 52)

Variables nuevas disponibles para el notebook 03:
   intensidad_festivo
   festivo_nacional
   festivo_parcial
   intensidad_vispera
   intensidad_post
   laborable_efectivo


---
## Qué ejecutar ahora

1. **Este cuaderno**, de principio a fin. Tarda unos segundos.
2. **El notebook 03 completo.** Recogerá automáticamente las variables nuevas,
   porque la función `clasificar_variables` incluye toda columna numérica que no
   sea un desfase corto.

No es necesario volver a ejecutar los notebooks 01 ni 02.

## Qué enviar para revisar

| Archivo | Para qué |
|---|---|
| `resultados/comparativa_ml.csv` | Ver si mejora el rendimiento global |
| `resultados/evaluacion_festivos.csv` | Lo importante: comparar la penalización |
| `resultados/festivos_detalle.csv` | Comprobar el 2 de mayo y Santiago en concreto |

## Qué esperar

**El rendimiento global apenas cambiará.** Los festivos son unos veinte días al
año, de modo que su peso sobre la métrica media es reducido. Si el MAPE se
mantiene en torno al 1,6 %, es el resultado correcto.

**Lo relevante estará en los festivos.** Las dos cifras a vigilar son el MAPE del
2 de mayo (9,25 % antes) y el de Santiago Apóstol (2,85 % antes). Si bajan, la
corrección ha funcionado.

**Revisa también la importancia de variables.** Si `intensidad_festivo` aparece
por encima de `festivo` en la sección 6, el modelo está aprovechando el matiz
nuevo, lo que constituye una confirmación adicional.